# Comparación aislada de R · EKF G/F con solubilidad variable

**Este notebook cambia únicamente R.** No ajusta Q, no elige fuentes de ruido y no recalibra química ni gas. El notebook 01 estudia estructuras/amplitudes de Q y ajusta R conjuntamente; sus optimizaciones no se usan aquí.

Se comparan dos valores **constantes**:

| Caso | Desviación estándar [g/(L h)] | R [(g/(L h))²] |
|---|---:|---:|
| Referencia actual | 0,04 | 0,0016 |
| Escenario del manual: ±4 % FS, rectangular, referencia de 2 L | 0,046695 | 0,00218044 |

La interpretación del ±4 % como fondo de escala y la distribución rectangular son **supuestos explícitos**, no afirmaciones del fabricante. La sección 5 del manual no especifica la base del porcentaje. El segundo caso no es el 0,0027 de la envolvente empírica del notebook 01.

Se mantiene R constante para aislar esta comparación. El volumen real sigue usándose al convertir las mediciones; la incertidumbre del segundo caso se referencia a 2 L y no se adapta automáticamente a las retiradas de volumen. Más abajo mostramos cuánto cambiaría el escenario al usar los volúmenes reales, sin añadir una tercera configuración ni confundirla con las dos ejecutadas.

**Datos:** LAB019–021 se muestran como desarrollo (participaron en calibrar la capa de gas) y LAB016–018 como holdout de la calibración de gas. Los seis se evalúan con parámetros congelados: no se ajustan ni seleccionan fuentes o valores de R usando el holdout. La biomasa presenta limitaciones de medición. En LAB016–018 hay muestras de biomasa y densidad, pero no química postinicial de YAN, etanol o G/F individuales para calcular sus RMSE. Los resultados se guardan dentro de este notebook; no se exportan archivos ni se crea una carpeta de resultados.


Incluye además un barrido de sensibilidad de R desde 0,001 hasta 1000 veces el actual. Este barrido no cambia ni selecciona parámetros usando el holdout.


In [ ]:
from pathlib import Path
import sys,json,hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nbformat
from IPython.display import display,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/wine_fermentation').is_dir())
HERE=ROOT/'notebooks/EKF/kalman_analysis'
sys.path.insert(0,str(HERE))
import analysis_core as core
NAME='02_gf_variable_solubility_R_comparison'
PROTECTED={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
    for directory in ['notebooks','src','configs'] for p in (ROOT/directory).rglob('*')
    if p.is_file() and '__pycache__' not in p.parts and not p.name.startswith(NAME+'.')
    and 'results_R_comparison' not in p.parts}
INSTRUMENT=json.loads((ROOT/'configs/co2_instrument.json').read_text())
assert INSTRUMENT['flow_range_sccm']==[0.,50.] and INSTRUMENT['calibration_gas']=='air'
assert INSTRUMENT['co2_correction_factor']==.74
DATA,CHEM,AUDIT,_=core.prepare(ROOT)
assert tuple(DATA)==('LAB019','LAB020','LAB021') and set(CHEM.lab)==set(DATA)
DEVELOPMENT=tuple(DATA)
HOLDOUT=('LAB016','LAB017','LAB018')
LABS=DEVELOPMENT+HOLDOUT
ROLE={lab:('Development' if lab in DEVELOPMENT else 'Holdout') for lab in LABS}
# Reuse the same clock corrections, horizons and holdout metadata as EKF05.
# load() prepares observations and metadata; it does not calibrate gas or chemistry.
sys.path.insert(0,str(ROOT/'src'))
from wine_fermentation.models.historical.calibration import load
v=load(ROOT)
volume_rows=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','remaining_volume_L'])
extra_audit=[]
for lab in HOLDOUT:
    horizon=float(v.horizons[lab]);batch=v.batches[lab]
    events=list(batch.pulses['N'])
    if lab=='LAB017':events=[(min(t for t,dose in events),sum(dose for t,dose in events))]
    events=np.array([(round(t,10),dose) for t,dose in events],float)
    ev=volume_rows.loc[volume_rows.lab.eq(lab)&volume_rows.record_kind.eq('withdrawal_after')].sort_values('time_h')
    et=ev.time_h.to_numpy();vv=ev.remaining_volume_L.to_numpy()
    def volume(t):return np.r_[2.,vv][np.searchsorted(et,np.asarray(t),side='right')]
    raw=v.raw_co2[lab].copy();raw['bin']=np.ceil(raw.t_h/.5-1e-10).astype(int);rows=[]
    for end in np.arange(.5,horizon+1e-9,.5):
        sample=raw.loc[raw.bin.eq(round(end/.5))].dropna(subset=['flow_sccm'])
        gaps=np.diff(np.r_[end-.5,sample.t_h.to_numpy(),end])
        y=sample.flow_sccm.to_numpy()*(44.0095*60*.74/(1000*24.16))/volume(sample.t_h.to_numpy())
        rows.append((end,float(np.mean(y)) if len(y) else np.nan,len(sample)>=30 and gaps.max()<=2/60))
    sensor=pd.DataFrame(rows,columns=['t_h','measured','valid'])
    sensor['assimilate']=sensor.valid&sensor.measured.ge(.05)&sensor.valid.shift(fill_value=False)&sensor.measured.shift().ge(.05)
    rawt=v.raw_temperature[lab].copy();rawt['minute']=np.floor(rawt.t_h*60).astype(int)
    temp=rawt.groupby('minute')['T'].median().dropna();available=(temp.index.to_numpy()+1)/60;temps=temp.to_numpy(float)
    initial=np.array([batch.initials[k] for k in core.STATES[:5]]+[0.,0.]);initial[4]=10.010194;S0=initial[2]+initial[3]
    r=initial[2]/S0
    P0=np.diag([max(.03,.2*initial[0])**2,.015**2,r*r*25+25,(1-r)**2*25+25,4.,0.,0.]);P0[2,3]=P0[3,2]=r*(1-r)*25-25
    grid=np.unique(np.round(np.r_[np.arange(0,horizon,1/12),sensor.t_h,horizon,events[:,0]],10))
    doses=np.zeros(len(grid));obs=np.full(len(grid),np.nan);valid=np.zeros(len(grid),bool);update=np.zeros(len(grid),bool)
    for t,dose in events:doses[np.argmin(abs(grid-t))]+=dose
    for row in sensor.itertuples():
        j=np.argmin(abs(grid-row.t_h));assert abs(grid[j]-row.t_h)<1e-8
        obs[j]=row.measured;valid[j]=row.valid;update[j]=row.assimilate
    DATA[lab]=dict(grid=grid,doses=doses,y=obs,valid=valid,update=update,initial=initial,P0=P0,available=available,temperature=temps,S0=S0,sensor=sensor,horizon=horizon)
    extra_audit.append(dict(LAB=lab,horizon_h=horizon,windows=len(sensor),updates=int(sensor.assimilate.sum()),X0=initial[0],N0=initial[1],G0=initial[2],F0=initial[3],E0=initial[4],pulse_h=events[0,0],dose_g_L=events[0,1]))
# Only actual holdout biomass observations are appended; missing chemistry is not invented.
CHEM=pd.concat([CHEM,v.biomass.loc[v.biomass.lab.isin(HOLDOUT),['lab','t_h','X_viable_g_l']]],ignore_index=True)
DENSITY=v.density.loc[v.density.lab.isin(LABS)].copy()
AUDIT=pd.concat([AUDIT,pd.DataFrame(extra_audit)],ignore_index=True);AUDIT['role']=AUDIT.LAB.map(ROLE)
assert tuple(DATA)==LABS
display(AUDIT)


## Cálculo de los dos valores

La configuración está confirmada por el usuario: 0–50 SCCM, calibración con aire, factor a CO₂ 0,74 correcto. No se vuelve a solicitar esa información.

Para el escenario FS: $a_f=0.04(50)=2$ SCCM. Bajo distribución rectangular $u_f=a_f/\sqrt3$. A 2 L:

\[
c_{2L}=\frac{44.0095\,60\,0.74}{1000\,24.16\,(2)}=0.04043919,
\quad \sigma_y=c_{2L}\frac{2}{\sqrt3},
\quad R=\sigma_y^2.
\]

La exactitud no es automáticamente un ruido blanco: parte del error puede ser persistente entre ventanas. No reducimos este componente por raíz del número de muestras. No sumamos otra vez el ±1 % ni la repetibilidad. Esta es una prueba de sensibilidad de ingeniería, no una nueva certificación de R.

Referencias: [manual OMEGA, sección 5](../../../references/Sensor_omega/M2842.pdf), [NIST: incertidumbre tipo B](https://physics.nist.gov/cuu/Uncertainty/typeb.html).


In [ ]:
REFERENCE_VOLUME_L=2.0
conversion=44.0095*60*INSTRUMENT['co2_correction_factor']/(1000*24.16*REFERENCE_VOLUME_L)
sigma_datasheet=conversion*(.04*INSTRUMENT['flow_range_sccm'][1])/np.sqrt(3)
CASES={'Original R':.04,'4% FS scenario':float(sigma_datasheet)}
R_TABLE=pd.DataFrame([dict(case=k,sigma_g_L_h=v,R=v*v) for k,v in CASES.items()])
display(R_TABLE)
assert np.isclose(CASES['Original R']**2,.0016)
assert np.isclose(CASES['4% FS scenario']**2,.002180439553,rtol=1e-5)
display(pd.Series(core.ARCHIVED_GAS,index=['kLa [1/h]','Gas gain'],name='Unchanged archived gas parameters'))
display(pd.Series(core.BASE_SIGMA,index=core.SOURCE_NAMES,name='Unchanged Q diffusion amplitudes'))
print('Four active noise sources; mortality and maintenance amplitudes remain zero.')
print('P0, chemical parameters, input processing and correction masks are unchanged.')


## Ejecución sin optimización

Se reutilizan las ecuaciones y el integrador verificados del análisis, sin ejecutar su optimización ni su recalibración de gas. Se mantienen kLa=0,1500799102 h⁻¹ y ganancia=0,6460643588, que corresponden al EKF G/F variable existente. La regla de construcción de Q y sus amplitudes permanecen fijas; como L depende del estado, su valor numérico puede variar entre trayectorias.

CO₂ mostrado: media predicha antes de incorporar cada medición. Estados químicos: después de la actualización cuando corresponde; propagados entre actualizaciones. Química solo para evaluación, incluyendo las 12 muestras postiniciales de YAN y las de nutrición por LAB019–021. En LAB016–018 se evalúa biomasa con las muestras disponibles; no se inventan RMSE de química no medida. Se conserva la exclusión de G+F sustituido por cero.

In [ ]:
RESULTS={};metric_rows=[];checks=[]
for label,rsd in CASES.items():
    RESULTS[label]={}
    for lab in LABS:
        result=core.run(DATA[lab],Rsd=rsd)
        RESULTS[label][lab]=result
        metrics,info,acf=core.diagnostics(DATA,CHEM,lab,result)
        metrics['case']=label;metrics['role']=ROLE[lab];metric_rows.extend(metrics.to_dict('records'))
        info.update(case=label,role=ROLE[lab],acf=acf.tolist());checks.append(info)
METRICS=pd.DataFrame(metric_rows);CHECKS=pd.DataFrame(checks)
assert METRICS.loc[METRICS.variable.eq('N'),'n'].eq(12).all()
assert CHECKS.min_eigenvalue.min()>-1e-7
# Reproduce the frozen reference on all six LABs, without fitting.
previous=nbformat.read(ROOT/'notebooks/EKF/05_zenteno_co2_variable_solubility.executed.ipynb',as_version=4)
prior=None
for cell in previous.cells:
    for output in cell.get('outputs',[]):
        for line in output.get('text','').splitlines():
            if line.startswith('METRICS_VARIABLE_JSON '):prior=pd.DataFrame(json.loads(line.split(' ',1)[1]))
assert prior is not None
for row in METRICS.loc[METRICS.case.eq('Original R')].itertuples():
    if row.variable in ('G','F'):continue
    expected=prior.loc[prior.LAB.eq(row.LAB)&prior.variable.eq(row.variable)&prior.caso.eq('forecast' if row.variable=='CO2' else 'EKF_')]
    assert len(expected)==1 and abs(row.RMSE-expected.RMSE.iloc[0])<1e-7
print('Original estimator RMSE reproduced on LAB019–021 and holdout LAB016–018. No optimization performed.')


## Figuras: dos trayectorias por panel

Las leyendas y los RMSE están dentro de los gráficos. Las muestras iniciales permanecen visibles y no entran al RMSE postinicial. Las estimaciones químicas no se corrigen usando esas muestras.

Los puntos iniciales representan los valores utilizados para iniciar el estimador; en particular, E0=10,010194 g/L es el supuesto original del escenario sin medida inicial de etanol. Un punto inicial no proporciona validación química postinicial.


In [ ]:
COLORS={'Original R':'#0072B2','4% FS scenario':'#D55E00'}
STYLES={'Original R':'-','4% FS scenario':'--'}
FIGURE_COUNT=0
def compare(state,title):
    global FIGURE_COUNT
    factor=1000 if state=='N' else 1.
    unit='g/(L h)' if state=='CO2' else ('mg/L' if state=='N' else 'g/L')
    fig,axs=plt.subplots(2,3,figsize=(15,7.6))
    for ax,lab in zip(axs.flat,LABS):
        d=DATA[lab];errors=[]
        for label in CASES:
            f,P,dg=RESULTS[label][lab]
            if state=='CO2':
                mask=d['valid']&np.isfinite(dg[:,0]);tx=d['grid'][mask];pred=dg[mask,0]
            else:
                tx=d['grid'];pred=f[:,2]+f[:,3] if state=='GF' else f[:,core.STATES.index(state)]
            ax.plot(tx,pred*factor,color=COLORS[label],ls=STYLES[label],label=label,lw=1.5)
            row=METRICS.loc[METRICS.LAB.eq(lab)&METRICS.variable.eq(state)&METRICS.case.eq(label)]
            if len(row):errors.append(f"RMSE {label} = {row.RMSE.iloc[0]:.3f} {unit}")
        if state=='CO2':
            sensor=d['sensor'];ax.plot(sensor.t_h,sensor.measured,c='black',lw=.7,label='Samples')
        elif state!='Cd':
            col=core.CHEM_COLS[state]
            o=CHEM.loc[CHEM.lab.eq(lab)&CHEM.t_h.gt(0)&CHEM.t_h.le(d['horizon'])].dropna(subset=[col])
            if state=='GF':o=o.loc[~o.GF_zero_substitution.astype(str).str.lower().eq('true')]
            sample_label='G+F' if state=='GF' else 'Samples'
            ax.scatter(o.t_h,o[col]*factor,c='black',s=22,label=sample_label,zorder=5)
            initial=d['initial'][2]+d['initial'][3] if state=='GF' else d['initial'][core.STATES.index(state)]
            ax.scatter([0],[initial*factor],c='black',s=22,zorder=5)
            if state=='GF':
                density=DENSITY
                density=density.loc[density.lab.eq(lab)&density.t_h.between(0,d['horizon'])].dropna(subset=['density_g_l'])
                ax.scatter(density.t_h,2.5*(density.density_g_l-1000),marker='D',facecolors='none',edgecolors='#009E73',s=23,label='G+F from density',zorder=4)
        for t in d['grid'][d['doses']>0]:ax.axvline(t,c='gray',ls=':',lw=.8,label='N pulse')
        h,l=ax.get_legend_handles_labels();unique=dict(zip(l,h))
        ax.legend(unique.values(),unique.keys(),loc='best',fontsize=6.8,title='\n'.join(errors) or None,title_fontsize=7,framealpha=.9)
        ax.set(title=lab+' · '+ROLE[lab],xlabel='Time [h]',ylabel=title+' ['+unit+']');ax.grid(alpha=.2)
    context='forecast before correction' if state=='CO2' else 'state estimate'
    fig.suptitle(f'{title} · EKF G/F · {context} · only R changed',y=.995)
    fig.tight_layout(pad=.7,rect=(0,0,1,.95));FIGURE_COUNT+=1;plt.show()


In [ ]:
compare('CO2','CO₂')

In [ ]:
compare('X','Biomass')

In [ ]:
compare('N','YAN')

In [ ]:
compare('GF','Glucose + fructose')

In [ ]:
compare('E','Ethanol')

In [ ]:
compare('Cd','Dissolved CO₂')

In [ ]:
compare('G','Glucose')

In [ ]:
compare('F','Fructose')

## Comparación numérica y diagnósticos

El resumen separa desarrollo y holdout y promedia los RMSE por laboratorio dentro de cada grupo, no mezcla ambos conjuntos. Un menor RMSE químico no demuestra por sí solo una mejor incertidumbre. NIS y autocorrelación son diagnósticos aproximados bajo selección de ventanas y recortes; no una prueba metrológica de que R sea correcto.

In [ ]:
PER_LAB=METRICS.pivot(index=['role','LAB','variable','unit','n'],columns='case',values='RMSE')
PER_LAB['Change [%]']=100*(PER_LAB['4% FS scenario']/PER_LAB['Original R']-1)
display(PER_LAB.round(4))
SUMMARY=METRICS.groupby(['role','variable','unit','case']).RMSE.mean().unstack('case')
SUMMARY['Change [%]']=100*(SUMMARY['4% FS scenario']/SUMMARY['Original R']-1)
display(SUMMARY.round(4))
display(CHECKS.drop(columns='acf').round(4))
fig,axs=plt.subplots(2,3,figsize=(15,7))
for ax,lab in zip(axs.flat,LABS):
    for row in CHECKS.loc[CHECKS.LAB.eq(lab)].itertuples():
        ax.plot(range(1,11),row.acf,color=COLORS[row.case],ls=STYLES[row.case],marker='o',label=row.case)
    ax.axhline(0,c='gray',lw=.7);ax.set(title=lab+' · '+ROLE[lab],xlabel='Lag in 30-minute bins',ylabel='Innovation autocorrelation');ax.grid(alpha=.2);ax.legend(loc='best',fontsize=7)
fig.suptitle('Innovation temporal structure · only R changed',y=.995)
fig.tight_layout(pad=.7,rect=(0,0,1,.95));FIGURE_COUNT+=1;plt.show()


## Alcance del valor constante y conclusión

El factor de conversión utiliza el volumen real. En cambio, el R del escenario ejecutado se calculó a **2 L de referencia**. Si se propagara la misma especificación al volumen V, su valor sería $R(V)=R(2L)(2/V)^2$. La tabla siguiente muestra esa sensibilidad, pero no corresponde a un tercer filtro ejecutado.

La comparación puede mostrar cómo responde el estimador a una R mayor. No permite demostrar que ±4 % sea FS, que el error sea blanco o que una R constante represente todas las fases. No se toma una decisión sobre las fuentes de Q con este experimento.

## Barrido amplio de sensibilidad: R entre 0,001 y 1000 veces el actual

La comparación anterior cambia R solo por un factor aproximado de 1,36. Ahora probamos **seis órdenes de magnitud**: $R=0.0016\,f$, con $f$ entre $10^{-3}$ y $10^3$, y mantenemos también el escenario del manual en su posición exacta.

**El factor multiplica la varianza R, no la desviación estándar:** $\sigma=0.04\sqrt f$. Los extremos son pruebas de estrés numérico; no estimaciones respaldadas de la precisión del medidor. No se modifica Q, P0, la química, la capa de gas, los iniciales, los pulsos ni las ventanas/máscaras de actualización. No se realiza optimización ni se selecciona R con el holdout.

Para evitar superponer catorce trayectorias, las curvas resumen muestran todos los factores y las figuras temporales muestran solo el valor original y los dos extremos. No se ocultan recortes o trayectorias químicas anómalas. Se separan desarrollo y holdout y no se inventa RMSE de estados no medidos.


In [ ]:
R_ORIGINAL=CASES['Original R']**2
SPEC_FACTOR=CASES['4% FS scenario']**2/R_ORIGINAL
R_FACTORS=np.unique(np.r_[np.logspace(-3,3,13),SPEC_FACTOR])
SENS_CONFIG=pd.DataFrame(dict(R_factor=R_FACTORS,R=R_ORIGINAL*R_FACTORS,sigma=np.sqrt(R_ORIGINAL*R_FACTORS)))
display(SENS_CONFIG)
SENS_RESULTS={};sensitivity_rows=[];sensitivity_checks=[]
for multiple in R_FACTORS:
    R_value=R_ORIGINAL*multiple
    if multiple==1.:result_set=RESULTS['Original R']
    elif multiple==SPEC_FACTOR:result_set=RESULTS['4% FS scenario']
    else:result_set={lab:core.run(DATA[lab],Rsd=np.sqrt(R_value)) for lab in LABS}
    SENS_RESULTS[float(multiple)]=result_set
    for lab in LABS:
        result=result_set[lab]
        metrics,info,acf=core.diagnostics(DATA,CHEM,lab,result)
        metrics['role']=ROLE[lab];metrics['R_factor']=multiple;metrics['R']=R_value
        sensitivity_rows.extend(metrics.to_dict('records'))
        d=DATA[lab];dg=result[2];m=d['update']
        # H K = (S - R)/S for the scalar linear observation, before projection.
        observation_weight=1-R_value/dg[m,2]
        f=result[0]
        late=d['grid']>100
        rises=(np.diff(f[:,1])>1e-8)&(d['grid'][1:]>100)&(d['doses'][1:]==0)
        assert np.all(d['update'][1:][rises]),'Positive late YAN increment outside correction'
        info.update(role=ROLE[lab],R_factor=multiple,R=R_value,acf=acf.tolist(),
            late_YAN_max_mg_L=float(f[late,1].max()*1000),
            late_positive_YAN_corrections=int(rises.sum()),
            mean_observation_weight=float(np.mean(observation_weight)),
            fraction_updates_clipped=float(np.mean(dg[m,4]>0)))
        sensitivity_checks.append(info)
    print(f'Completed R factor {multiple:.4g} on all six LABs.')
SENS_METRICS=pd.DataFrame(sensitivity_rows);SENS_CHECKS=pd.DataFrame(sensitivity_checks)
assert len(R_FACTORS)==14
assert len(SENS_RESULTS)==14 and len(SENS_CHECKS)==14*6
assert SENS_CHECKS.min_eigenvalue.min()>-1e-7
assert SENS_METRICS.loc[SENS_METRICS.variable.eq('N'),'n'].eq(12).all()
assert set(SENS_METRICS.loc[SENS_METRICS.role.eq('Holdout'),'variable'])=={'CO2','X'}
SENS_MEAN=SENS_METRICS.groupby(['role','variable','unit','R_factor'],as_index=False).RMSE.mean()
base=SENS_MEAN.loc[SENS_MEAN.R_factor.eq(1),['role','variable','RMSE']].rename(columns={'RMSE':'reference_RMSE'})
SENS_MEAN=SENS_MEAN.merge(base,on=['role','variable'],validate='many_to_one')
SENS_MEAN['Change [%]']=100*(SENS_MEAN.RMSE/SENS_MEAN.reference_RMSE-1)
display(SENS_MEAN.pivot(index=['role','variable','unit'],columns='R_factor',values='RMSE').round(4))
display(SENS_MEAN.pivot(index=['role','variable','unit'],columns='R_factor',values='Change [%]').round(2))


### Curvas de respuesta a R

Cada línea resume el RMSE medio por laboratorio dentro de desarrollo o holdout. El sombreado es el **mínimo y máximo entre laboratorios**, no un intervalo de confianza. Los RMSE químicos usan las mismas muestras para todos los factores. Las líneas verticales sitúan el valor original y el escenario 4 % FS.


In [ ]:
ROLE_COLORS={'Development':'#0072B2','Holdout':'#D55E00'}
fig,axs=plt.subplots(2,3,figsize=(15,7.5))
for ax,state in zip(axs.flat,['CO2','X','N','GF','E','F']):
    part=SENS_METRICS.loc[SENS_METRICS.variable.eq(state)]
    unit=part.unit.iloc[0]
    for role,group in part.groupby('role',sort=False):
        stats=group.groupby('R_factor').RMSE.agg(['mean','min','max']).sort_index()
        xx=stats.index.to_numpy();color=ROLE_COLORS[role]
        ax.semilogx(xx,stats['mean'],marker='o',ms=3,color=color,label=role+' mean')
        ax.fill_between(xx,stats['min'].to_numpy(),stats['max'].to_numpy(),color=color,alpha=.12)
    ax.axvline(1,c='black',ls=':',lw=.9,label='Original R')
    ax.axvline(SPEC_FACTOR,c='gray',ls='--',lw=.9,label='4% FS scenario')
    ax.set(title=state,xlabel='R / original R (variance multiplier)',ylabel='RMSE ['+unit+']')
    ax.grid(alpha=.2);ax.legend(loc='best',fontsize=6.5)
fig.suptitle('R sensitivity · means and LAB ranges · no parameter fitting',y=.995)
fig.tight_layout(pad=.7,rect=(0,0,1,.95));FIGURE_COUNT+=1;plt.show()


### Innovaciones, peso de corrección y restricciones

El peso de corrección mostrado es $HK=1-R/S$, donde S es la varianza de innovación calculada por el filtro. Describe la corrección lineal de la observación acumulada antes del recorte y del reinicio de la ventana; **no es la ganancia de cada estado químico**. Una R pequeña no garantiza mejores estados. La proporción de actualizaciones con algún recorte y el NIS ayudan a detectar el coste de forzar el seguimiento. El NIS no constituye por sí solo una prueba chi-cuadrado exacta bajo selección de datos y proyecciones.


In [ ]:
fig,axs=plt.subplots(1,3,figsize=(15,4))
for ax,col,title in zip(axs,['mean_NIS','mean_observation_weight','fraction_updates_clipped'],
                       ['Mean NIS','Mean linear observation correction weight','Updates with physical clipping']):
    for role,g in SENS_CHECKS.groupby('role',sort=False):
        agg=g.groupby('R_factor')[col].agg(['mean','min','max']).sort_index();xx=agg.index.to_numpy();color=ROLE_COLORS[role]
        ax.semilogx(xx,agg['mean'],marker='o',ms=3,label=role+' mean',color=color)
        ax.fill_between(xx,agg['min'].to_numpy(),agg['max'].to_numpy(),color=color,alpha=.12)
    ax.axvline(1,c='black',ls=':',lw=.9,label='Original R');ax.axvline(SPEC_FACTOR,c='gray',ls='--',lw=.9,label='4% FS scenario')
    ax.set(title=title,xlabel='R / original R (variance multiplier)');ax.grid(alpha=.2);ax.legend(loc='best',fontsize=6.5)
axs[0].set_yscale('log');axs[0].set_ylabel('NIS (log scale)')
axs[1].set_ylabel('H K');axs[1].set_ylim(0,1.03)
axs[2].set_ylabel('Fraction of eligible updates');axs[2].set_ylim(0,1.03)
fig.suptitle('R sensitivity · innovation and correction diagnostics',y=.995)
fig.tight_layout(pad=.7,rect=(0,0,1,.95));FIGURE_COUNT+=1;plt.show()
display(SENS_CHECKS.drop(columns='acf').round(5))


### Trayectorias con los extremos del barrido

Se muestran tres casos: R × 0,001, R original y R × 1000. Esos factores equivalen a aproximadamente 0,032 y 31,6 veces la desviación estándar original. Las observaciones, iniciales y nutriciones son las mismas; únicamente cambia R. La ausencia de muestras químicas postiniciales en el holdout impide interpretar sus trayectorias no medidas como una validación.


In [ ]:
# Reuse the existing plotting policy without altering the original two-case results.
SAVED_COMPARISON=(CASES,RESULTS,COLORS,STYLES,METRICS)
try:
    EXTREMES={'R x 0.001':.001,'Original R':1.,'R x 1000':1000.}
    CASES={label:np.sqrt(R_ORIGINAL*factor) for label,factor in EXTREMES.items()}
    RESULTS={label:SENS_RESULTS[float(factor)] for label,factor in EXTREMES.items()}
    COLORS={'R x 0.001':'#D55E00','Original R':'#0072B2','R x 1000':'#009E73'}
    STYLES={'R x 0.001':':','Original R':'-','R x 1000':'--'}
    parts=[]
    for label,factor in EXTREMES.items():
        part=SENS_METRICS.loc[SENS_METRICS.R_factor.eq(factor)].copy();part['case']=label;parts.append(part)
    METRICS=pd.concat(parts,ignore_index=True)
    for state,title in [('CO2','CO₂'),('X','Biomass'),('N','YAN'),('GF','Glucose + fructose'),('E','Ethanol')]:compare(state,title)
finally:
    CASES,RESULTS,COLORS,STYLES,METRICS=SAVED_COMPARISON


### Lectura del barrido

Estas curvas muestran sensibilidad, no un ajuste ni un nuevo valor elegido para R. En particular, no se utiliza el mínimo observado en LAB016–018 para escoger una configuración. Un mejor RMSE de gas que empeore los estados, aumente los recortes o deje innovaciones incompatibles con la incertidumbre declarada no respalda una mejora general del estimador.

Se conservan las hipótesis y limitaciones de la comparación inicial: error temporalmente independiente como aproximación, amplitudes de Q sin reajustar y datos químicos limitados. Todos los resultados de este barrido permanecen dentro del notebook, sin archivos auxiliares de resultados.


In [ ]:
EXTREME_SUMMARY=SENS_MEAN.loc[SENS_MEAN.R_factor.isin([.001,1.,1000.])]
display(EXTREME_SUMMARY[['role','variable','unit','R_factor','RMSE','Change [%]']].round(4))
for role in ['Development','Holdout']:
    text=[]
    for state in ['CO2','X','N','GF','E']:
        g=SENS_MEAN.loc[SENS_MEAN.role.eq(role)&SENS_MEAN.variable.eq(state)]
        if len(g):
            text.append(f"{state}: RMSE entre {g.RMSE.min():.4g} y {g.RMSE.max():.4g} {g.unit.iloc[0]}")
    display(Markdown('**Rango observado en '+role+'**: '+'; '.join(text)+'. Los rangos no seleccionan un valor de R.'))
assert not (HERE/'results_R_comparison').exists()

LATE_YAN=SENS_CHECKS.loc[SENS_CHECKS.R_factor.isin([.001,1.,1000.]),['role','LAB','R_factor','late_YAN_max_mg_L','late_positive_YAN_corrections']]
display(LATE_YAN.round(3))
display(Markdown('**Alerta de interpretación:** los incrementos de YAN después de 100 h y sin pulso coinciden con correcciones de CO₂. No provienen de la producción de nitrógeno en el modelo, cuyo balance entre nutriciones es de consumo. En LAB016–018 no hay muestras postiniciales de YAN para validarlos; se muestran como anomalías del estimador, no como una mejora.'))


In [ ]:
display(pd.DataFrame([dict(reference_volume_L=v,sigma=CASES['4% FS scenario']*2/v,R=(CASES['4% FS scenario']*2/v)**2) for v in [2.,1.8,1.75]]))
display(Markdown('**Comparación descriptiva con parámetros congelados.** Las tablas separan desarrollo y holdout. No se ajustan ni se seleccionan parámetros o fuentes de ruido con LAB016–018. No se escriben tablas ni archivos de resultados fuera del notebook.'))
assert all(hashlib.sha256((ROOT/path).read_bytes()).hexdigest()==digest for path,digest in PROTECTED.items())
assert FIGURE_COUNT==16
assert set(METRICS.loc[METRICS.role.eq('Holdout'),'variable'])=={'CO2','X'}
verification=dict(allowed_labs=list(LABS),only_R_parameter_changed=True,Q_amplitudes=core.BASE_SIGMA.tolist(),
    gas_parameters=core.ARCHIVED_GAS.tolist(),R_values={k:v*v for k,v in CASES.items()},
    sensitivity_variance_factors=R_FACTORS.tolist(),comparison_volume_reference_L=REFERENCE_VOLUME_L,figures=FIGURE_COUNT,existing_files_unchanged=len(PROTECTED))
display(pd.Series(verification,name="Verification"))
print('Existing notebooks, helpers, configurations and previous analysis results unchanged:',len(PROTECTED))
print('No fit or source selection; LAB019–021 and holdout LAB016–018 evaluated with frozen parameters.')
